# Process weather data

In [1]:
from pathlib import Path
from solhycool_evaluation.utils import preprocess_meteonorm_txt_data
import pandas as pd

%load_ext autoreload
%autoreload 2

data_path: Path = Path("../../data")
meteo_data_file_path: Path = data_path / "datasets/tmy_2005_guadix.txt"
output_path: Path = Path("../results")

df_env = preprocess_meteonorm_txt_data(meteo_data_file_path)
df_env.head()


2025-03-31 14:56:46.604 | INFO     | solhycool_evaluation.utils:preprocess_meteonorm_txt_data:42 - Pre-processed data saved to ../../data/datasets/tmy_2005_guadix.csv and tmy_2005_guadix.h5


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td
time,,,,,,,,,,
2005-01-01 00:00:00+00:00,0,0,0,0,6.9,69,0.0,-142.2,0.0,1.7
2005-01-01 01:00:00+00:00,0,0,0,0,6.1,69,0.0,-166.2,0.0,1.0
2005-01-01 02:00:00+00:00,0,0,0,0,5.3,74,0.0,-126.1,0.0,1.1
2005-01-01 03:00:00+00:00,0,0,0,0,4.9,78,0.0,-106.3,0.0,1.5
2005-01-01 04:00:00+00:00,0,0,0,0,4.5,84,0.0,-94.4,0.0,2.1


In [2]:
# Visualize data
from plotly_resampler import FigureWidgetResampler
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from phd_visualizations.constants import generate_plotly_config

df_env = pd.read_hdf(meteo_data_file_path.with_suffix(".h5"))

var_ids: list[str] = ["Tamb_C", "HR_pct", "precip_mm"]
var_units: list[str] = ["°C", "%", "mm"]

fig = make_subplots(rows=len(var_ids), cols=1, shared_xaxes=True)
fig = FigureWidgetResampler(fig)

for i, (var_id, var_unit) in enumerate(zip(var_ids, var_units)):
    fig.add_trace(
        go.Scattergl(name=var_id, showlegend=True), 
        hf_x=df_env.index, 
        hf_y=df_env[var_id], 
        # max_n_samples=2_000,
        row=i + 1, col=1
    )
    fig.update_yaxes(title_text=f"{var_id.replace("_", " ")} ({var_unit})", row=i + 1)

fig.update_layout(
    height=650,
    width=800,
    title="<b>Weather variables</b>",
    title_x=0.1,
    legend_traceorder="normal",
    legend=dict(orientation="h", y=1.08, xanchor="left", x=0),
)

fig

# fig.show(
#     config=generate_plotly_config(
#         fig, figure_name=f"solhycool_env_viz_{df_env.index[0].strftime('%Y%m%d')}"
#     )
# )



FigureWidgetResampler({
    'data': [{'name': '<b style="color:sandybrown">[R]</b> Tamb_C <i style="color:#fc9944">~9h</i>',
              'showlegend': True,
              'type': 'scattergl',
              'uid': '81f3f7f0-d140-42c4-b841-728ae53e586e',
              'x': array([datetime.datetime(2005, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 1, 1, 7, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 1, 1, 14, 0, tzinfo=datetime.timezone.utc), ...,
                          datetime.datetime(2005, 12, 31, 7, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 12, 31, 15, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 12, 31, 23, 0, tzinfo=datetime.timezone.utc)],
                         shape=(1000,), dtype=object),
              'xaxis': 'x',
              'y': array([ 6.9,  3.5, 13.4, ...,  0.6, 10.8,  3.7], shape=(1000,)),
     

In [3]:
from phd_visualizations import save_figure

start, end = fig.layout.xaxis.range
save_figure(fig, f"solhycool_weather_viz_{start[:10].replace('-', '')}_{end[:10].replace('-', '')}", figure_path=output_path, formats=["png", "svg"])


TypeError: cannot unpack non-iterable NoneType object